In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

import ollama 
import json
import requests

In [2]:
@tool
def web_search(query: str, max_result=5) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.
    Input: query string
    Output: formatted JSON string
    """
    response = ollama.web_search(query, max_results=max_result)
    return response.results

response = web_search.invoke({"query": "best travel destinations in Europe"})

c:\Users\laxmi\anaconda3\envs\ml\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


In [3]:
response

[WebSearchResult(content='[Skip to main content](https://www.cntraveler.com/www.cntraveler.com#main-content)\n\nSave this story\n\nSave this story\n\nAs we edge closer to the end of each year, we begin to look ahead to the destinations we’re most looking forward to visiting (and recommending to you, our fellow travelers) for the following 12 months. We’ve asked our expert contributors from all around the globe to nominate the spots that are on the up—the places that are on their radar thanks to a flock of hotel openings, major cultural moments, talk of new flight routes, or concerted conservation efforts taking root. These nominations make up the [Best Places to Go in 2026](https://www.cntraveler.com/story/the-best-places-to-go-in-2026)—the places worthy of your precious annual leave and hard-earned spending money.\n\nFor the second year running, in addition to turning our gaze to global destinations in the [Best Places to Go in the World in 2026](https://www.cntraveler.com/story/the-b

In [ ]:
import requests
from langchain_core.tools import tool
import os

@tool
def get_weather(location: str) -> str:
    """Get current weather for a location using WeatherAPI.com.
    
    Use for queries about weather, temperature, or conditions in any city.
    Examples: "weather in Paris", "temperature in Tokyo", "is it raining in London"
    
    Args:
        location: City name (e.g., "New York", "London", "Tokyo")
        
    Returns:
        Current weather information including temperature and conditions.
    """

    url = (
        "http://api.weatherapi.com/v1/current.json"
        f"?key={os.getenv("WEATHER_API_KEY")}&q={location}&aqi=no"
    )

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()
    return data


# Test
response = get_weather.invoke({"location": "Paris"})
print(response)


ReadTimeout: HTTPSConnectionPool(host='wttr.in', port=443): Read timed out. (read timeout=10)

In [ ]:
response

In [ ]:
import subprocess
import sys

@tool
def hotel_search(query):
    """Search for hotels using Airbnb MCP async function."""

    code = f"""
import asyncio
from airbnb_mcp import hotel_search
asyncio.run(hotel_search("{query}"))
"""
    
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)

    return result.stdout

In [ ]:
query = "show me the best hotels in mumbai"
response = hotel_search.invoke({"query": query})

print(response)

Loaded 2 Tools
Tools Available: [StructuredTool(name='airbnb_search', description='Search for Airbnb listings with various filters and pagination. Provide direct links to the user', args_schema={'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'Location to search for (city, state, etc.)'}, 'placeId': {'type': 'string', 'description': 'Google Maps Place ID (overrides the location parameter)'}, 'checkin': {'type': 'string', 'description': 'Check-in date (YYYY-MM-DD)'}, 'checkout': {'type': 'string', 'description': 'Check-out date (YYYY-MM-DD)'}, 'adults': {'type': 'number', 'description': 'Number of adults'}, 'children': {'type': 'number', 'description': 'Number of children'}, 'infants': {'type': 'number', 'description': 'Number of infants'}, 'pets': {'type': 'number', 'description': 'Number of pets'}, 'minPrice': {'type': 'number', 'description': 'Minimum price for the stay'}, 'maxPrice': {'type': 'number', 'description': 'Maximum price for the stay'}, 'cur

### Travel Agent with Google Gemini

In [ ]:
# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:
# Combine all tools
tools = [web_search, get_weather, hotel_search]

In [ ]:
system_prompt = """You are an expert Travel Planning Agent with access to real-time web search, weather data, and hotel search capabilities.

YOUR MISSION: Create comprehensive, personalized travel itineraries that help users plan memorable trips.

AVAILABLE TOOLS:
1. web_search(query, max_result) - Search for attractions, restaurants, activities, travel tips, and local information
2. get_weather(location) - Get current weather and forecast for destinations
3. hotel_search(query) - Search for hotels and accommodations on Airbnb

PLANNING PROCESS - Follow these steps systematically:

STEP 1: UNDERSTAND THE REQUEST
- Extract destination(s)
- Determine trip duration (default: 5 days if not specified)
- Identify preferences (budget, interests, travel style)
- Note any special requirements (dietary, accessibility, etc.)

STEP 2: GATHER DESTINATION INTELLIGENCE
- Search for "best things to do in [destination]"
- Search for "travel guide [destination] 2025"
- Search for "[destination] local tips hidden gems"
- Get current weather conditions for the destination

STEP 3: RESEARCH ACCOMMODATIONS
- Search for hotels matching the user's preferences and budget
- Include different areas/neighborhoods if applicable
- Consider proximity to attractions

STEP 4: BUILD THE ITINERARY
Create a day-by-day plan with:
- Morning, afternoon, and evening activities
- Recommended restaurants and dining options
- Transportation tips between locations
- Time estimates and distances
- Mix of popular attractions and local experiences

STEP 5: ADD PRACTICAL INFORMATION
- Weather conditions and what to pack
- Estimated budget breakdown
- Best times to visit attractions
- Local customs and tips
- Transportation options

STEP 6: PRESENT THE PLAN
Format your response as:

# Travel Plan for [Destination]
**Duration:** [X] days
**Best Time:** [Current weather insights]

## Day 1: [Theme/Focus]
### Morning (9:00 AM - 12:00 PM)
- **Activity:** [Name and description]
- **Location:** [Address/Area]
- **Why visit:** [Brief explanation]
- **Tips:** [Practical advice]

### Afternoon (12:00 PM - 6:00 PM)
- **Lunch:** [Restaurant recommendation]
- **Activity:** [Name and description]
- **Tips:** [Practical advice]

### Evening (6:00 PM - 10:00 PM)
- **Dinner:** [Restaurant recommendation]
- **Activity:** [Optional evening activity]

[Repeat for each day]

## Accommodation Recommendations
[List 3-5 hotel options with prices, ratings, and links]

## Weather & Packing
[Current weather and packing suggestions]

## Budget Estimate
- Accommodation: [Range]
- Food: [Estimated daily cost]
- Activities: [Estimated costs]
- Transportation: [Estimated costs]
- **Total:** [Estimated total]

## Local Tips
- [Transportation advice]
- [Cultural customs]
- [Safety tips]
- [Money/payment tips]

## Useful Links
[Include relevant sources and booking links]

IMPORTANT RULES:
✓ Always perform at least 3-4 web searches to gather comprehensive information
✓ Check weather to provide seasonal advice
✓ Include hotel options with direct links
✓ Balance popular attractions with authentic local experiences
✓ Provide specific names, addresses, and timing
✓ Include budget estimates
✓ Consider travel time between locations
✓ Add insider tips and recommendations
✓ Use current, real-time information
✓ Make the itinerary realistic and achievable
✓ If days not specified, create a 5-day plan
✓ Cite your sources with URLs

BE CREATIVE AND THOUGHTFUL: Don't just list attractions. Create a narrative flow for each day that makes sense geographically and thematically. Think like a local tour guide who wants to show visitors the best of your city.

Now, help the user plan an amazing trip!"""

In [ ]:
# Create the travel planning agent
travel_agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

### Test the Travel Planning Agent

In [ ]:
from langchain_core.messages import HumanMessage

# Example query - you can modify this
query = "Plan a 3-day trip to Paris for a couple interested in art, food, and history"

# Invoke the agent
result = travel_agent.invoke({'messages': [HumanMessage(content=query)]})

ConnectTimeout: HTTPSConnectionPool(host='wttr.in', port=443): Max retries exceeded with url: /Paris?format=j1 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000014564364470>, 'Connection to wttr.in timed out. (connect timeout=10)'))

In [ ]:
# Display the travel plan
print(result['messages'][-1].content[0]['text'])